In [1]:
import pandas as pd
import numpy as np
from scipy.stats import randint as sp_randint
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import RandomizedSearchCV
from sklearn.neighbors import KNeighborsClassifier

# --- Загрузка исходных файлов ---
# Читаем тренировочный набор
raw_train = pd.read_csv('/kaggle/input/competitions/titanic/train.csv')
# Читаем тестовый набор
raw_test = pd.read_csv('/kaggle/input/competitions/titanic/test.csv')
# Читаем образец_submission для сохранения в правильном формате
submission_form = pd.read_csv('/kaggle/input/competitions/titanic/gender_submission.csv')


In [2]:
# --- Подготовка объединенного датасета ---
# Создаем копии для безопасной работы
train_flagged = raw_train.copy()
test_flagged = raw_test.copy()

# Добавляем метку, чтобы потом разделить обратно (1 - тренировка, 0 - тест)
train_flagged['is_train'] = 1
test_flagged['is_train'] = 0

# Объединяем train и test в один пул данных для общей обработки признаков
union_data = pd.concat(
    [train_flagged, test_flagged],
    ignore_index=True,    # Сброс индексов
    axis=0,               # Конкатенация по строкам
    sort=False            # Сохранение порядка колонок
)

In [3]:
# --- Функция создания новых признаков ---
def create_features(df: pd.DataFrame, test: bool = False) -> pd.DataFrame:
    # 1. Обработка титулов (Title) из имени
    df['Title'] = df['Name'].str.extract(r'([A-Za-z]+)\.', expand=False)

    # Группировка редких титулов в основные категории
    title_mapping = {'Mlle': 'Miss', 'Major': 'Mr', 'Col': 'Mr', 'Sir': 'Mr', 'Don': 'Mr', 
                     'Mme': 'Miss', 'Jonkheer': 'Mr', 'Lady': 'Mrs', 'Capt': 'Mr', 
                     'Countess': 'Mrs', 'Ms': 'Miss', 'Dona': 'Mrs'}
    df['Title'] = df['Title'].map(title_mapping).fillna(df['Title'])
    
    # Заполнение пропусков в возрасте медианой по группе титулов
    df['Age'] = df.groupby('Title')['Age'].transform(lambda x: x.fillna(x.median()))

    # 2. Размер семьи
    df['Family_Size'] = df['Parch'] + df['SibSp']

    # 3. Извлечение фамилии
    df['Last_Name'] = df['Name'].apply(lambda x: str.split(x, ",")[0])
    
    # Заполнение пропусков в цене билета средним
    df['Fare'] = df['Fare'].fillna(df['Fare'].mean())
    
    # 4. Логика выживания семьи (Family_Survival)
    DEFAULT_SURVIVAL_VALUE = 0.5
    df['Family_Survival'] = DEFAULT_SURVIVAL_VALUE

    # Группировка по Фамилии и цене билета
    for grp, grp_df in df[['Survived','Name', 'Last_Name', 'Fare', 'Ticket', 'PassengerId',
                            'SibSp', 'Parch', 'Age', 'Cabin']].groupby(['Last_Name', 'Fare']):
        
        if (len(grp_df) != 1):
            for ind, row in grp_df.iterrows():
                # Максимальное и минимальное выживание в группе (без текущего пассажира)
                smax = grp_df.drop(ind)['Survived'].max()
                smin = grp_df.drop(ind)['Survived'].min()
                passID = row['PassengerId']
                
                if (smax == 1.0):
                    df.loc[df['PassengerId'] == passID, 'Family_Survival'] = 1
                elif (smin == 0.0):
                    df.loc[df['PassengerId'] == passID, 'Family_Survival'] = 0

    # Группировка по номеру билета (Ticket)
    for _, grp_df in df.groupby('Ticket'):
        if (len(grp_df) != 1):
            for ind, row in grp_df.iterrows():
                # Обновляем только если значение еще не определено четко
                if (row['Family_Survival'] == 0) | (row['Family_Survival'] == 0.5):
                    smax = grp_df.drop(ind)['Survived'].max()
                    smin = grp_df.drop(ind)['Survived'].min()
                    passID = row['PassengerId']
                    
                    if (smax == 1.0):
                        df.loc[df['PassengerId'] == passID, 'Family_Survival'] = 1
                    elif (smin == 0.0):
                        df.loc[df['PassengerId'] == passID, 'Family_Survival'] = 0

    # 5. Биннинг (разбиение на интервалы) числовых признаков
    df['Fare'] = df['Fare'].fillna(df['Fare'].median())
    
    # Квантильное разбиение Fare и Age
    df['FareBin'] = pd.qcut(df['Fare'], 5)
    df['FareBin_Code'] = df['FareBin'].cat.codes

    df['AgeBin'] = pd.qcut(df['Age'], 4)
    df['AgeBin_Code'] = df['AgeBin'].cat.codes

    # 6. Кодирование пола
    df['Sex'] = df['Sex'].map({'male': 0, 'female': 1})

    # 7. Удаление неиспользуемых колонок
    df = df.drop(columns=['Name', 'PassengerId', 'SibSp', 'Parch', 'Ticket', 'Cabin',
                  'Embarked', 'Fare', 'Age', 'Title', 'Last_Name', 'FareBin', 'AgeBin'])

    return df

# Применяем функцию ко всему объединенному датасету
union_data = create_features(union_data)

In [4]:
# --- Восстановление разделения на train и test ---
# Выбираем строки, помеченные как тренировочные
train_final = union_data[union_data['is_train'] == 1].drop('is_train', axis=1)
# Выбираем строки, помеченные как тестовые (убираем таргет Survived, если есть)
test_final = union_data[union_data['is_train'] == 0].drop(['is_train', 'Survived'], axis=1)

# --- Подготовка матриц для модели ---
model_X = train_final.drop('Survived', axis=1)
model_y = train_final['Survived']
model_X_test = test_final

# --- Масштабирование признаков (StandardScaler) ---
scaler_std = StandardScaler()
model_X = scaler_std.fit_transform(model_X)
model_X_test = scaler_std.transform(model_X_test)

In [5]:
# --- Настройка RandomizedSearchCV для KNN ---
# Варианты параметров для случайной выборки
neighbor_options = [6, 7, 8, 9, 10, 11, 12, 14, 16, 18, 20, 22]
algo_options = ['auto']
weight_options = ['uniform', 'distance']
# Для leaf_size можно использовать диапазон, из которого будут случайно выбираться значения
leaf_options = list(range(1, 50, 5))

# Словарь параметров для поиска
param_space_random = {
    'algorithm': algo_options, 
    'weights': weight_options, 
    'leaf_size': leaf_options, 
    'n_neighbors': neighbor_options
}

# Инициализация случайного поиска по гиперпараметрам
# n_iter=30 означает, что будет проверено 30 случайных комбинаций
random_search_cv = RandomizedSearchCV(
    estimator=KNeighborsClassifier(), 
    param_distributions=param_space_random, 
    n_iter=30,                # Количество итераций подбора
    scoring="roc_auc",        # Метрика качества
    cv=10,                    # Количество фолдов кросс-валидации
    verbose=True,             # Показывать процесс обучения
    random_state=42,          # Фиксация случайности для воспроизводимости
    n_jobs=-1                 # Использование всех доступных ядер процессора
)

# Запуск подбора параметров
random_search_cv.fit(model_X, model_y)

# Вывод результатов поиска
print(f"Лучшая метрика (ROC-AUC): {random_search_cv.best_score_:.4f}")
print(f"Оптимальные параметры: {random_search_cv.best_params_}")
print(f"Лучшая модель: {random_search_cv.best_estimator_}")

Fitting 10 folds for each of 30 candidates, totalling 300 fits
Лучшая метрика (ROC-AUC): 0.8791
Оптимальные параметры: {'weights': 'uniform', 'n_neighbors': 18, 'leaf_size': 21, 'algorithm': 'auto'}
Лучшая модель: KNeighborsClassifier(leaf_size=21, n_neighbors=18)


In [6]:
# --- Финальный прогноз ---
# Используем лучшую найденную модель
best_model = random_search_cv.best_estimator_
best_model.fit(model_X, model_y)

# Предсказание на тестовых данных
pred_values = best_model.predict(model_X_test)

# --- Подготовка файла submission ---
submission_form['Survived'] = pred_values.astype(int)
submission_form.to_csv('submission.csv', index=False)